# Stable-profit validation

Weekly valuation, young access, sizing and a conservative shortlist verdict.

In [1]:
from pathlib import Path
import sys
import json
import math
import hashlib
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd() / "scratchpad/hyperliquid-ic" if (Path.cwd() / "scratchpad/hyperliquid-ic").exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
ARTIFACTS = PROJECT_DIR / "_artifacts-stable-profit"
ARTIFACTS.mkdir(exist_ok=True)
REWRITE = PROJECT_DIR / "_artifacts-rewrite"
from ic_research import ResearchConfig, completed_daily_window, simulate_allocator, summarise_backtest, daily_feature_ic, write_json
from stable_profit import classification_metrics, paired_block_sharpe_samples, short_core_columns, simulate_stable_policy, walk_forward_loss_predictions, weekly_curve

CONFIG = ResearchConfig()
features = pd.read_parquet(REWRITE / "features.parquet")
labels = pd.read_parquet(REWRITE / "labels.parquet")
observations = pd.read_parquet(REWRITE / "observations.parquet")
evaluation_start, evaluation_end = completed_daily_window(observations)
features = features[features.date.between(evaluation_start - pd.Timedelta(days=CONFIG.training_history_days), evaluation_end)].copy()
labels = labels[labels.date.isin(features.date)].copy()
evaluation = features[features.date.between(evaluation_start, evaluation_end)].copy()
print("Stable-profit panel:", features.shape, "| dates:", features.date.nunique(), "| evaluation:", evaluation_start, "to", evaluation_end)


Stable-profit panel: (104618, 410) | dates: 545 | evaluation: 2025-09-13 00:00:00 to 2026-09-12 00:00:00


In [2]:
primary_metrics = pd.read_csv(ARTIFACTS / "primary-policy-metrics.csv", index_col="arm")
curves = {arm: pd.read_parquet(ARTIFACTS / f"equity-{arm}.parquet") for arm in ["A0", "A1", "A2", "A3"]}
weekly_rows = []
for arm, curve in curves.items():
    weekly = weekly_curve(curve)
    weekly.to_parquet(ARTIFACTS / f"weekly-equity-{arm}.parquet", index=False)
    row = summarise_backtest(weekly, periods_per_year=52.0)
    row["arm"] = arm
    weekly_rows.append(row)
weekly_metrics = pd.DataFrame(weekly_rows).set_index("arm")
weekly_metrics.to_csv(ARTIFACTS / "weekly-policy-metrics.csv")
display(primary_metrics[["cagr", "volatility", "sharpe", "max_drawdown", "mean_cash_fraction", "mean_stale_positions", "mean_carried_positions"]])
display(weekly_metrics[["cagr", "volatility", "sharpe", "max_drawdown"]])
classifier_metrics = json.loads((ARTIFACTS / "loss-classifier-metrics.json").read_text())
family_classifier_metrics = json.loads((ARTIFACTS / "family-loss-classifier-metrics.json").read_text())
a3_trades = pd.read_parquet(ARTIFACTS / "trades-A3.parquet")
a3_active = a3_trades[a3_trades.veto_active]
a3_active_start = a3_active.date.min() if not a3_active.empty else pd.NaT
a3_active_end = a3_active.date.max() if not a3_active.empty else pd.NaT
btc_reference = pd.read_parquet(REWRITE / "btc-reference.parquet")
btc_reference.index = pd.to_datetime(btc_reference.index).tz_localize(None).normalize()
btc_reference = btc_reference[~btc_reference.index.duplicated(keep="last")].sort_index()
btc_regime = pd.DataFrame({"date": btc_reference.index, "btc_return": btc_reference.close.pct_change().to_numpy()})
btc_regime["btc_regime"] = np.select(
    [btc_regime.btc_return.gt(0), btc_regime.btc_return.lt(0)],
    ["up", "down"],
    default=pd.NA,
)
regime_rows = []
for arm, curve in curves.items():
    joined = curve.merge(btc_regime[["date", "btc_regime"]], on="date", how="inner")
    for regime, subset in joined.dropna(subset=["btc_regime"]).groupby("btc_regime"):
        returns = subset["return"].dropna()
        deviation = returns.std(ddof=1)
        regime_rows.append({
            "arm": arm,
            "btc_regime": regime,
            "start": str(subset.date.min()),
            "end": str(subset.date.max()),
            "dates": int(subset.date.nunique()),
            "return_observations": int(len(returns)),
            "mean_return": float(returns.mean()) if len(returns) else np.nan,
            "volatility": float(deviation * math.sqrt(365.0)) if pd.notna(deviation) else np.nan,
            "sharpe": float(returns.mean() / deviation * math.sqrt(365.0)) if pd.notna(deviation) and deviation > 0 else np.nan,
            "mean_positions": float(subset.n_positions.mean()),
            "mean_cash_fraction": float((subset.cash / subset.equity).mean()),
            "mean_stale_positions": float(subset.stale_positions.mean()),
            "mean_carried_positions": float(subset.carried_positions.mean()),
        })
pd.DataFrame(regime_rows).to_csv(ARTIFACTS / "btc-regime-policy-metrics.csv", index=False)


,cagr,volatility,sharpe,max_drawdown,mean_cash_fraction,mean_stale_positions,mean_carried_positions
arm,,,,,,,
A0,0.049487,0.200354,0.340885,-0.156379,0.024776,0.0,1.972603
A1,0.028863,0.162362,0.256041,-0.085517,0.027761,0.0,1.967123
A2,-0.069938,0.161353,-0.368957,-0.149066,0.029058,0.0,1.972603
A3,-0.037065,0.166777,-0.143275,-0.103255,0.021666,0.0,1.969863


,cagr,volatility,sharpe,max_drawdown
arm,,,,
A0,0.049627,0.217306,0.328818,-0.148643
A1,0.028943,0.152878,0.260572,-0.071968
A2,-0.070123,0.155019,-0.389926,-0.129569
A3,-0.037165,0.166549,-0.143790,-0.086773


In [3]:
features_young = features.copy()
features_young["young_proxy"] = features_young.interval_log_growth_30 / features_young.observed_covered_days_30.replace(0, np.nan)
features_young["young_proxy"] = features_young["young_proxy"].combine_first(features_young.interval_log_growth_14 / features_young.observed_covered_days_14.replace(0, np.nan)).combine_first(features_young.interval_log_growth_7 / features_young.observed_covered_days_7.replace(0, np.nan))
features_young["young_score"] = features_young.incumbent_score

# Freeze the empirical mapping at each month boundary and give dates equal
# weight.  The CDF and quantile arrays are precomputed once per month; doing a
# groupby for every young row makes the otherwise small diagnostic needlessly
# expensive on the 100k-row panel.
for period in sorted(features_young.date.dt.to_period("M").unique()):
    month_start = pd.Timestamp(period.start_time)
    prior = features_young[features_young.date < month_start]
    current = features_young.date.dt.to_period("M").eq(period) & features_young.available_history_days.lt(90) & features_young.incumbent_score.isna()
    if not current.any():
        continue
    proxy_groups = [np.sort(group.to_numpy(dtype=float)) for _, group in prior.dropna(subset=["young_proxy"]).groupby("date")["young_proxy"] if len(group)]
    score_groups = [group.to_numpy(dtype=float) for _, group in prior.dropna(subset=["incumbent_score"]).groupby("date")["incumbent_score"] if len(group)]
    if prior.dropna(subset=["young_proxy"]).date.nunique() < 20 or prior.dropna(subset=["incumbent_score"]).date.nunique() < 20 or not proxy_groups or not score_groups:
        continue
    indices = features_young.index[current]
    values = features_young.loc[indices, "young_proxy"].to_numpy(dtype=float)
    valid = np.isfinite(values)
    if not valid.any():
        continue
    ranks = np.zeros(valid.sum(), dtype=float)
    for group in proxy_groups:
        ranks += np.searchsorted(group, values[valid], side="right") / len(group)
    ranks /= len(proxy_groups)
    mapped = np.zeros(len(ranks), dtype=float)
    for group in score_groups:
        mapped += np.quantile(group, np.clip(ranks, 0, 1))
    mapped /= len(score_groups)
    mapped[values[valid] <= 0] = 0.0
    features_young.loc[indices[valid], "young_score"] = mapped
features_young["young_score"] = features_young["young_score"].fillna(0.0)
features_young = features_young[features_young.date.between(evaluation_start, evaluation_end)].copy()
predictions = pd.read_parquet(ARTIFACTS / "loss-predictions-30.parquet")
a2_scores = predictions[["date", "address", "loss_probability"]].rename(columns={"loss_probability": "veto_score"})
young_results, young_trades, young_pools = {}, {}, {}
for name, scores, veto in [("A0-young", None, None), ("A2-young", a2_scores, "veto_score")]:
    curve, trade, pool = simulate_stable_policy(features_young, observations, policy_name=name, veto_scores=scores, config=CONFIG, max_positions=6, veto_feature=veto, veto_fraction=.20, young_extension=True)
    young_results[name], young_trades[name], young_pools[name] = curve, trade, pool
    curve.to_parquet(ARTIFACTS / f"equity-{name}.parquet", index=False)
    trade.to_parquet(ARTIFACTS / f"trades-{name}.parquet", index=False)
    pool.to_parquet(ARTIFACTS / f"pool-{name}.parquet", index=False)

# Match A0-young's final capped allocation to A2-young's realised deployment
# fraction so cash dilution is separated from the loss-veto effect.
a2_deployment = young_trades["A2-young"].set_index("date")["invested_fraction"]
cash_curve, cash_trade, cash_pool = simulate_stable_policy(
    features_young, observations, policy_name="A0-young-cash-matched", config=CONFIG,
    max_positions=6, young_extension=True, matched_invested_fraction_by_date=a2_deployment,
)
young_results["A0-young-cash-matched"] = cash_curve
young_trades["A0-young-cash-matched"], young_pools["A0-young-cash-matched"] = cash_trade, cash_pool
cash_curve.to_parquet(ARTIFACTS / "equity-A0-young-cash-matched.parquet", index=False)
cash_trade.to_parquet(ARTIFACTS / "trades-A0-young-cash-matched.parquet", index=False)
cash_pool.to_parquet(ARTIFACTS / "pool-A0-young-cash-matched.parquet", index=False)
young_metrics = pd.DataFrame([dict(summarise_backtest(curve), arm=name) for name, curve in young_results.items()]).set_index("arm")
young_metrics.to_csv(ARTIFACTS / "young-policy-metrics.csv")
display(young_metrics[["cagr", "volatility", "sharpe", "max_drawdown", "mean_cash_fraction"]])
young_attribution = young_pools["A0-young"].merge(
    features[["date", "address", "is_sparse_prediction", "last_interval_days"]],
    on=["date", "address"], how="left",
).merge(labels[["date", "address", "forward_log_growth_raw_30"]], on=["date", "address"], how="left")
young_attribution["age_cohort"] = pd.cut(
    young_attribution.available_history_days,
    [-np.inf, 30, 90, 180, 360, np.inf],
    labels=["<30", "30-89", "90-179", "180-359", "360+"], right=False,
)
young_attribution["sampling_cohort"] = np.select(
    [young_attribution.is_sparse_prediction.fillna(False), young_attribution.last_interval_days.gt(2)],
    ["weekly-plus", "irregular"], default="daily",
)
selected_young = young_attribution[young_attribution.selected].copy()
selected_young["gross_pnl_proxy"] = selected_young.target_dollars * np.expm1(selected_young.forward_log_growth_raw_30)
young_selection_summary = selected_young.groupby(["age_cohort", "sampling_cohort"], observed=False).agg(
    selected_rows=("address", "size"),
    selected_vaults=("address", "nunique"),
    target_dollars=("target_dollars", "sum"),
    gross_pnl_proxy=("gross_pnl_proxy", "sum"),
    loss_probability=("forward_log_growth_raw_30", lambda s: float((s < math.log(.9)).mean())),
).reset_index()
young_selection_summary.to_csv(ARTIFACTS / "young-selection-attribution.csv", index=False)

# Dynamic accepted-dollar sensitivity: cap each arm at its own current
# pre-cap requested allocation, so greedy redistribution cannot create a new
# concentration as equity changes.
cap_results, cap_trades, cap_pools = {}, {}, {}
for name, scores, veto in [("A0-cap", None, None), ("A2-cap", a2_scores, "veto_score")]:
    curve, trade, pool = simulate_stable_policy(
        evaluation, observations, policy_name=name, veto_scores=scores,
        config=CONFIG, max_positions=6, veto_feature=veto, veto_fraction=.20,
        cap_at_requested_allocation=True,
    )
    cap_results[name], cap_trades[name], cap_pools[name] = curve, trade, pool
    curve.to_parquet(ARTIFACTS / f"equity-{name}.parquet", index=False)
    trade.to_parquet(ARTIFACTS / f"trades-{name}.parquet", index=False)
    pool.to_parquet(ARTIFACTS / f"pool-{name}.parquet", index=False)

# Sizing cash-matched A0: scale final capped weights towards A2-cap's realised
# investment fraction for the same A0 ranking and dynamic cap policy.
a2_cap_deployment = cap_trades["A2-cap"].set_index("date")["invested_fraction"]
cash_curve, cash_trade, cash_pool = simulate_stable_policy(
    evaluation, observations, policy_name="A0-cap-cash-matched", config=CONFIG,
    max_positions=6, cap_at_requested_allocation=True,
    matched_invested_fraction_by_date=a2_cap_deployment,
)
cap_results["A0-cap-cash-matched"], cap_trades["A0-cap-cash-matched"], cap_pools["A0-cap-cash-matched"] = cash_curve, cash_trade, cash_pool
cash_curve.to_parquet(ARTIFACTS / "equity-A0-cap-cash-matched.parquet", index=False)
cash_trade.to_parquet(ARTIFACTS / "trades-A0-cap-cash-matched.parquet", index=False)
cash_pool.to_parquet(ARTIFACTS / "pool-A0-cap-cash-matched.parquet", index=False)

# Contributor removal is a diagnostic, not a new candidate.  Remove the
# largest cumulative selected-capital address from the beginning and rerun.
leader = cap_pools["A0-cap"].groupby("address").target_dollars.sum().idxmax()
leader_curve, leader_trade, leader_pool = simulate_stable_policy(
    evaluation[evaluation.address.ne(leader)].copy(), observations,
    policy_name="A0-cap-without-leader", config=CONFIG, max_positions=6,
    cap_at_requested_allocation=True,
)
cap_results["A0-cap-without-leader"], cap_trades["A0-cap-without-leader"], cap_pools["A0-cap-without-leader"] = leader_curve, leader_trade, leader_pool
leader_curve.to_parquet(ARTIFACTS / "equity-A0-cap-without-leader.parquet", index=False)
leader_trade.to_parquet(ARTIFACTS / "trades-A0-cap-without-leader.parquet", index=False)
leader_pool.to_parquet(ARTIFACTS / "pool-A0-cap-without-leader.parquet", index=False)
a2_leader = cap_pools["A2-cap"].groupby("address").target_dollars.sum().idxmax()
a2_leader_curve, a2_leader_trade, a2_leader_pool = simulate_stable_policy(
    evaluation[evaluation.address.ne(a2_leader)].copy(), observations,
    policy_name="A2-cap-without-leader", veto_scores=a2_scores, config=CONFIG,
    max_positions=6, veto_feature="veto_score", veto_fraction=.20,
    cap_at_requested_allocation=True,
)
cap_results["A2-cap-without-leader"], cap_trades["A2-cap-without-leader"], cap_pools["A2-cap-without-leader"] = a2_leader_curve, a2_leader_trade, a2_leader_pool
a2_leader_curve.to_parquet(ARTIFACTS / "equity-A2-cap-without-leader.parquet", index=False)
a2_leader_trade.to_parquet(ARTIFACTS / "trades-A2-cap-without-leader.parquet", index=False)
a2_leader_pool.to_parquet(ARTIFACTS / "pool-A2-cap-without-leader.parquet", index=False)
cap_metrics = pd.DataFrame([dict(summarise_backtest(curve), arm=name) for name, curve in cap_results.items()]).set_index("arm")
cap_metrics.to_csv(ARTIFACTS / "accepted-dollar-cap-metrics.csv")
display(cap_metrics[["cagr", "volatility", "sharpe", "max_drawdown", "mean_cash_fraction"]])
def cash_match_diagnostics(trade, reference):
    joined = trade[["date", "invested_fraction", "matched_invested_fraction"]].merge(
        reference[["date", "invested_fraction"]].rename(columns={"invested_fraction": "reference_invested_fraction"}),
        on="date", how="inner",
    )
    gap = joined.invested_fraction - joined.reference_invested_fraction
    positive_reference = joined.reference_invested_fraction.gt(1e-9)
    return {
        "dates": int(len(joined)),
        "actual_mean_invested_fraction": float(joined.invested_fraction.mean()),
        "reference_mean_invested_fraction": float(joined.reference_invested_fraction.mean()),
        "mean_gap": float(gap.mean()),
        "mean_absolute_gap": float(gap.abs().mean()),
        "cap_limited_dates": int((positive_reference & gap.lt(-1e-9)).sum()),
        "matched_dates": int((positive_reference & gap.abs().le(1e-9)).sum()),
        "zero_reference_dates": int((~positive_reference).sum()),
    }
write_json(ARTIFACTS / "cap-diagnostic-manifest.json", {
    "leader_address": leader,
    "candidate_leader_address": a2_leader,
    "cap_definition": "current arm pre-cap requested weights times current investable budget",
    "cash_match_reference": "A2-cap realised invested fraction",
    "cash_match_method": "scale final capped weights down when A0 cannot exceed the reference; cap-limited dates remain below it",
    "cap_cash_match": cash_match_diagnostics(cap_trades["A0-cap-cash-matched"], cap_trades["A2-cap"]),
    "young_cash_match": cash_match_diagnostics(young_trades["A0-young-cash-matched"], young_trades["A2-young"]),
})


,cagr,volatility,sharpe,max_drawdown,mean_cash_fraction
arm,,,,,
A0-young,0.085020,0.212373,0.489893,-0.151628,0.075197
A2-young,-0.071322,0.184616,-0.308726,-0.151306,0.060470
A0-young-cash-matched,0.080511,0.210602,0.472467,-0.152408,0.091662


,cagr,volatility,sharpe,max_drawdown,mean_cash_fraction
arm,,,,,
A0-cap,0.124750,0.105360,1.168005,-0.063917,0.452635
A2-cap,0.037001,0.097713,0.420213,-0.088777,0.441338
A0-cap-cash-matched,0.129221,0.098868,1.278203,-0.048398,0.474202
A0-cap-without-leader,0.029726,0.116907,0.308374,-0.131126,0.433627
A2-cap-without-leader,-0.029413,0.106292,-0.228193,-0.124347,0.427893


In [4]:
uncertainty = []
for block in [44, 88]:
    samples_by_arm, point_by_arm, dates_by_arm = {}, {}, {}
    for arm in ["A1", "A2", "A3"]:
        point, samples, dates = paired_block_sharpe_samples(curves[arm], curves["A0"], block_days=block, seed=10_000 + block)
        point_by_arm[arm], samples_by_arm[arm], dates_by_arm[arm] = point, samples, dates
    assert len(set(dates_by_arm.values())) == 1, dates_by_arm
    raw_matrix = np.column_stack([samples_by_arm[arm] for arm in ["A1", "A2", "A3"]])
    joint_finite = np.isfinite(raw_matrix).all(axis=1)
    matrix = raw_matrix[joint_finite]
    valid_draws = len(matrix)
    if valid_draws:
        points = np.asarray([point_by_arm[arm] for arm in ["A1", "A2", "A3"]], dtype=float)
        simultaneous_radius = float(np.quantile(np.max(np.abs(matrix - points), axis=1), .95))
    else:
        matrix, simultaneous_radius = np.empty((0, 3)), np.nan
    for column, arm in enumerate(["A1", "A2", "A3"]):
        point = point_by_arm[arm]
        samples = matrix[:, column] if valid_draws else np.array([], dtype=float)
        paired = curves[arm][["date", "return"]].merge(curves["A0"][["date", "return"]], on="date", suffixes=("_left", "_right")).dropna()
        differences = paired.return_left.to_numpy() - paired.return_right.to_numpy()
        uncertainty.append({
            "arm": arm,
            "block_days": block,
            "dates": dates_by_arm[arm],
            "blocks": math.ceil(dates_by_arm[arm] / block) if dates_by_arm[arm] else 0,
            "valid_resamples": len(samples),
            "mean_return_difference": float(differences.mean()) if len(differences) else np.nan,
            "mean_sharpe_difference": point,
            "bootstrap_p05": float(np.quantile(samples, .05)) if len(samples) else np.nan,
            "bootstrap_p95": float(np.quantile(samples, .95)) if len(samples) else np.nan,
            "simultaneous_p05": float(point - simultaneous_radius) if np.isfinite(simultaneous_radius) else np.nan,
            "simultaneous_p95": float(point + simultaneous_radius) if np.isfinite(simultaneous_radius) else np.nan,
        })
pd.DataFrame(uncertainty).to_csv(ARTIFACTS / "paired-block-uncertainty.csv", index=False)

def counterfactual_curve(curve, variant):
    source = curve.sort_values("date").copy()
    original_returns = source["return"].fillna(0.0)
    if variant == "remove-best-day":
        keep = source.index
        adjusted = original_returns.copy()
        best_index = original_returns.iloc[1:].idxmax() if len(original_returns) > 1 else None
        if best_index is not None:
            adjusted.loc[best_index] = 0.0
    elif variant == "exclude-February":
        keep = source.index[source.date.dt.month != 2]
        adjusted = original_returns.loc[keep]
    else:
        raise ValueError(variant)
    result = source.loc[keep].copy()
    result["return"] = adjusted.to_numpy(dtype=float)
    result["equity"] = float(source.equity.iloc[0]) * (1 + result["return"]).cumprod()
    result["drawdown"] = result.equity / result.equity.cummax() - 1
    return result

def summarise_counterfactual(curve):
    result = summarise_backtest(curve)
    observations = curve["return"].dropna()
    years = max(len(observations) / 365.0, 1 / 365.0)
    result["effective_years"] = years
    result["cagr"] = float((curve.equity.iloc[-1] / curve.equity.iloc[0]) ** (1 / years) - 1)
    return result

falsification = []
for arm, curve in {**curves, **young_results, **cap_results}.items():
    for variant in ["remove-best-day", "exclude-February"]:
        counterfactual = counterfactual_curve(curve, variant)
        falsification.append({"arm": arm, "variant": variant, **summarise_counterfactual(counterfactual)})
falsification_table = pd.DataFrame(falsification)
falsification_table.to_csv(ARTIFACTS / "falsification-sensitivities.csv", index=False)


In [5]:
rows = []
def excess_sharpe(curve, periods_per_year=365.0, annual_reference=0.05):
    returns = curve["return"].dropna().to_numpy(dtype=float)
    if len(returns) < 2:
        return np.nan
    volatility = returns.std(ddof=1) * np.sqrt(periods_per_year)
    return float((returns.mean() * periods_per_year - annual_reference) / volatility) if volatility > 0 else np.nan

excess_rows = []
for arm, curve in curves.items():
    excess_rows.append({"arm": arm, "clock": "daily", "excess_sharpe_5pct": excess_sharpe(curve)})
for arm in curves:
    excess_rows.append({"arm": arm, "clock": "weekly", "excess_sharpe_5pct": excess_sharpe(weekly_curve(pd.read_parquet(ARTIFACTS / f"equity-{arm}.parquet")), periods_per_year=52.0)})
excess_sharpe_table = pd.DataFrame(excess_rows)
excess_sharpe_table.to_csv(ARTIFACTS / "excess-sharpe-5pct.csv", index=False)
for arm in ["A1", "A2", "A3"]:
    candidate, anchor = primary_metrics.loc[arm], primary_metrics.loc["A0"]
    rows.append({"arm": arm, "higher_sharpe": bool(candidate.sharpe > anchor.sharpe), "no_higher_volatility": bool(candidate.volatility <= anchor.volatility), "no_deeper_drawdown": bool(candidate.max_drawdown >= anchor.max_drawdown), "cagr_floor_10pct": bool(candidate.cagr >= .10), "within_5pp_of_anchor": bool(candidate.cagr >= anchor.cagr - .05), "point_estimate_pass": bool(candidate.sharpe > anchor.sharpe and candidate.volatility <= anchor.volatility and candidate.max_drawdown >= anchor.max_drawdown and candidate.cagr >= .10 and candidate.cagr >= anchor.cagr - .05)})
verdict = pd.DataFrame(rows).set_index("arm")
verdict.to_csv(ARTIFACTS / "shortlist-verdict.csv")
matched_control = pd.read_csv(ARTIFACTS / "matched-finite-veto-rows.csv")
null_control = pd.read_csv(ARTIFACTS / "veto-permutation-null.csv")
matched_dates = pd.to_datetime(matched_control["date"])
common_pool = evaluation.merge(
    labels[["date", "address", "forward_log_growth_raw_30"]], on=["date", "address"], how="inner"
)
common_pool = common_pool[
    common_pool.forward_log_growth_raw_30.notna()
    & common_pool.interval_downside_dev_30.notna()
].copy()
finite_common = common_pool.dropna(subset=["btc_beta_30"])
common_dense_fraction = float((common_pool.last_interval_days > 2).mean())
finite_dense_fraction = float((finite_common.last_interval_days > 2).mean())
endpoint_errors = pd.to_numeric(evaluation["btc_endpoint_error_days_30"], errors="coerce").dropna().abs()
endpoint_median_hours = float(endpoint_errors.median() * 24) if not endpoint_errors.empty else np.nan
endpoint_max_hours = float(endpoint_errors.max() * 24) if not endpoint_errors.empty else np.nan
control_summary = {
    "A1": (matched_control.a1_tail_growth.mean(), matched_control.a1_tail_loss_probability.mean()),
    "A2": (matched_control.a2_tail_growth.mean(), matched_control.a2_tail_loss_probability.mean()),
    "A3": (matched_control.a3_tail_growth.mean(), matched_control.a3_tail_loss_probability.mean()),
}
null_growth = float(null_control.excluded_mean_growth.mean())
null_loss = float(null_control.excluded_loss_probability.mean())
falsification_lookup = falsification_table.set_index(["arm", "variant"])
a0_best_day = falsification_lookup.loc[("A0", "remove-best-day")]
a0_ex_february = falsification_lookup.loc[("A0", "exclude-February")]
classifier_text = (
    f"The A2 classifier is used rank-only for vetoing (ROC AUC {classifier_metrics.get('roc_auc', np.nan):.3f}, "
    f"PR AUC {classifier_metrics.get('pr_auc', np.nan):.3f}); its log loss {classifier_metrics.get('log_loss', np.nan):.3f} "
    f"versus base-rate {classifier_metrics.get('base_rate_log_loss', np.nan):.3f} is not a calibrated probability claim. "
    f"The optional family bundle is scored separately (ROC AUC {family_classifier_metrics.get('roc_auc', np.nan):.3f}) in `family-loss-classifier-metrics.json`."
)
source_hashes = {
    name: hashlib.sha256((PROJECT_DIR / name).read_bytes()).hexdigest()
    for name in ["ic_research.py", "stable_profit.py", "build_stable_profit_notebooks.py"]
}
write_json(ARTIFACTS / "run-manifest.json", {"plan": "stable-profit-plan-01", "primary_horizon": 30, "primary_arms": ["A0", "A1", "A2", "A3"], "veto_fraction": .20, "min_train_dates": 60, "logistic_C": 1.0, "decision_frequency": "daily", "daily_block_lengths": [44, 88], "config": CONFIG.__dict__, "source_hashes": source_hashes, "note": "Existing year is retrospective development evidence; no production promotion."})
summary = PROJECT_DIR / "summary-03.md"
summary.write_text(
    "# Stable-profit experiment results\n\n"
    f"The stable-profit track preserved the production return ranking and compared fixed 30-day tail vetoes: observed interval downside (A1), a sparse-compatible severe-loss classifier (A2), and endpoint-aligned absolute BTC beta (A3). A0 is the production replay anchor. The BTC beta rebuild uses actual sparse interval endpoints with a 12-hour reference tolerance; rows with aligned pairs have a median endpoint error of about {endpoint_median_hours:.1f} hours and a maximum of about {endpoint_max_hours:.1f} hours, all within tolerance. This removes the earlier dense-period-only artefact.\n\n"
    "## Primary results\n\n"
    + primary_metrics[["cagr", "volatility", "sharpe", "max_drawdown", "mean_cash_fraction", "mean_stale_positions", "mean_carried_positions"]].round(4).to_markdown()
    + "\n\nWeekly metrics use recomputed weekly equity returns and 52 periods per year:\n\n"
    + weekly_metrics[["cagr", "volatility", "sharpe", "max_drawdown"]].round(4).to_markdown()
    + f"\n\nA3's BTC-beta veto was active on {len(a3_active)} dates from {a3_active_start.date() if pd.notna(a3_active_start) else 'n/a'} to {a3_active_end.date() if pd.notna(a3_active_end) else 'n/a'}; its full-period metric includes the earlier A0 replay window.\n\n"
    + classifier_text
    + "\n\n"
    + "\n\n## Young and sizing checks\n\n"
    + young_metrics[["cagr", "volatility", "sharpe", "max_drawdown", "mean_cash_fraction"]].round(4).to_markdown()
    + "\n\nYoung access, age and sampling attribution (including selected capital and forward-loss P&L proxies) is saved in `young-selection-attribution.csv`. Young arms apply the 5% cap to under-90-day names and to names with intervals longer than two days; short proxy spans are labelled in the young pool ledgers.\n\n"
    + "\n\nThe cash-scaled A0-young arm scales final capped weights down towards the realised A2-young decision-time investment fraction, separating cash dilution from selection where the young caps permit it. Cap-limited dates remain below the reference and are recorded in `cap-diagnostic-manifest.json`. The sizing diagnostic applies each arm's current pre-cap requested allocation before redistribution:\n\n"
    + cap_metrics[["cagr", "volatility", "sharpe", "max_drawdown", "mean_cash_fraction"]].round(4).to_markdown()
    + "\n\nA0-cap's cash-heavy result is a sizing/exposure diagnostic, not a production candidate; the leader-removal and cash-scaled controls are saved with the cap artefacts. Cash matching is cap-feasible rather than exact when A0's TVL or per-vault limits bind.\n\n"
    "## Uncertainty and falsification\n\n"
    + f"Paired 44/88-day daily-return block bootstraps include pointwise and simultaneous Sharpe-difference bounds in `_artifacts-stable-profit/paired-block-uncertainty.csv`. The A0 remove-best-day counterfactual has CAGR {a0_best_day.cagr:.3f}, while excluding February has CAGR {a0_ex_february.cagr:.3f}; these are attribution sensitivities, not new tuning targets. The 5% annual-reference excess-Sharpe sensitivity is saved in `excess-sharpe-5pct.csv`, and conditional-return volatility/Sharpe by BTC up/down regime is saved in `btc-regime-policy-metrics.csv`.\n\n"
    + f"On identical finite rows, the comparison has {len(matched_control)} dates with at least one excluded row from {matched_dates.min().date()} to {matched_dates.max().date()} (the finite BTC-beta intersection begins {finite_common.date.min().date() if not finite_common.empty else 'n/a'}). Its long-interval share is {finite_dense_fraction:.1%}, versus {common_dense_fraction:.1%} in the full evaluation pool, so this is a near-daily subset rather than a complete sparse comparison. Mean excluded growth/loss rates are A1 {control_summary['A1'][0]:.3f}/{control_summary['A1'][1]:.1%}, A2 {control_summary['A2'][0]:.3f}/{control_summary['A2'][1]:.1%}, and A3 {control_summary['A3'][0]:.3f}/{control_summary['A3'][1]:.1%}, versus permutation-null {null_growth:.3f}/{null_loss:.1%}; the learned veto does not beat the simple downside control.\n\n"
    "## Decision\n\n"
    + verdict.to_markdown()
    + f"\n\nThe policy replay forward-fills each vault's latest causally available NAV, matching the production universe's daily candle convention. `mean_carried_positions` identifies the exposure marked from such carried values; it does not reveal an unobserved intra-interval path. The no-op A0 replay remains exactly equal to the incumbent path, but neither replay is yet identical to the trade-executor engine. No production promotion follows without unseen-time validation and exact engine reconciliation.\n"
)
display(verdict)
print(summary)


,higher_sharpe,no_higher_volatility,no_deeper_drawdown,cagr_floor_10pct,within_5pp_of_anchor,point_estimate_pass
arm,,,,,,
A1,False,True,True,False,True,False
A2,False,True,True,False,False,False
A3,False,True,True,False,False,False


/Users/moo/code/getting-started/scratchpad/hyperliquid-ic/summary-03.md
